### DeepSeek-R1-Distill-Qwen-1.5B token embeddings

Load the model from https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B

In [1]:
import torch
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B"
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16 if device == "cuda" else torch.float32,
    attn_implementation="eager",    # needed to get attention weights back
).to(device)
model.eval()

C:\Users\Dell\OneDrive\Documents\TomDocuments\PythonProjects\DL\20261001\DeepLearning\DL_MATH70116_2026\dl_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 339/339 [00:06<00:00, 49.34it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 1536)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=1536, out_features=1536, bias=True)
          (k_proj): Linear(in_features=1536, out_features=256, bias=True)
          (v_proj): Linear(in_features=1536, out_features=256, bias=True)
          (o_proj): Linear(in_features=1536, out_features=1536, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (up_proj): Linear(in_features=1536, out_features=8960, bias=False)
          (down_proj): Linear(in_features=8960, out_features=1536, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((1536,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((1536,), eps=1e-06)
    (rotar

In [2]:
input_sentence = "quantitative investment" #
input_sentence = "Investors seek shelter from AI rout in asset-heavy stocks" #"Hello, how are you doing today?"
input_sentence = "US software and private capital shares hit with fresh wave of selling"
input_sentence = "consumer discretionary stocks rout"
input_sentence = "The collateralization of cryptocurrency derivatives caused unprecedented deleveraging across interconnected counterparties"
tokens = tokenizer.tokenize(input_sentence)
print(tokens)

['The', 'Ġcollateral', 'ization', 'Ġof', 'Ġcryptocurrency', 'Ġderivatives', 'Ġcaused', 'Ġunprecedented', 'Ġdele', 'ver', 'aging', 'Ġacross', 'Ġinterconnected', 'Ġcounterpart', 'ies']


In [3]:
# BPE tokens carry a "Ġ" marker for a leading space; strip it for readable labels
#raw_tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
cleaned_tokens = [t.replace("Ġ", "") for t in tokens]

In [4]:
print(cleaned_tokens)

['The', 'collateral', 'ization', 'of', 'cryptocurrency', 'derivatives', 'caused', 'unprecedented', 'dele', 'ver', 'aging', 'across', 'interconnected', 'counterpart', 'ies']


In [5]:
emb_layer = model.get_input_embeddings()
embeddings = emb_layer.weight              # shape (vocab_size, hidden_size)
print(embeddings.shape, embeddings.dtype)           # (151936, 1536)
total_num_tokens = embeddings.shape[0]
embedding_vector_size = embeddings.shape[1]
print("total_num_tokens = ", total_num_tokens)
print("embedding_vector_size = ", embedding_vector_size)

torch.Size([151936, 1536]) torch.float32
total_num_tokens =  151936
embedding_vector_size =  1536


In [6]:
# get the full vocabulary of tokens
vocab = tokenizer.get_vocab()  # returns a dict of {token_string: token_id}
print(type(vocab), len(vocab))

<class 'dict'> 151665


### Count the trainable parameters

In [7]:
# total number of trainable parameters:
total = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total:,}")

Total trainable parameters: 1,777,088,000


In [15]:
def count(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

In [16]:
rows = [("embed_tokens", count(model.model.embed_tokens))]
rows += [(f"layer {i}", count(l)) for i, l in enumerate(model.model.layers)]
rows += [("final norm", count(model.model.norm))]
if model.lm_head.weight.data_ptr() != model.model.embed_tokens.weight.data_ptr():
    rows.append(("lm_head", count(model.lm_head)))
else:
    rows.append(("lm_head (tied)", 0))

for name, n in rows:
    print(f"{name:<16s} {n:>15,}")
print(f"{'TOTAL':<16s} {sum(n for _, n in rows):>15,}")

embed_tokens         233,373,696
layer 0               46,797,824
layer 1               46,797,824
layer 2               46,797,824
layer 3               46,797,824
layer 4               46,797,824
layer 5               46,797,824
layer 6               46,797,824
layer 7               46,797,824
layer 8               46,797,824
layer 9               46,797,824
layer 10              46,797,824
layer 11              46,797,824
layer 12              46,797,824
layer 13              46,797,824
layer 14              46,797,824
layer 15              46,797,824
layer 16              46,797,824
layer 17              46,797,824
layer 18              46,797,824
layer 19              46,797,824
layer 20              46,797,824
layer 21              46,797,824
layer 22              46,797,824
layer 23              46,797,824
layer 24              46,797,824
layer 25              46,797,824
layer 26              46,797,824
layer 27              46,797,824
final norm                 1,536
lm_head   